# 03 — Link prediction sanity (crude)

Compare a handful of **true edges** to **random non-edges** using a distance heuristic in raw ``Inf.Pos`` space. This is **not** a proper Mercator likelihood or held-out evaluation — only a sanity check that nearby pairs are somewhat enriched for edges.

**Setup and feature matrix**

Loads coordinates and the graph, then stacks `Inf.Pos` as `X`. `dist_sphere` measures angle between **directions** after normalizing each row of `X` (scale-invariant). `dist_raw` is kept as an optional Euclidean distance in unnormalized coordinates.

In [ ]:
import os

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 200  # default 100; sharper inline figures
import networkx as nx
import numpy as np
import pandas as pd

import dmercator_io as dm

RUN_SUBDIR = "d2"
RUN_SUBDIR = os.environ.get("DMERCATOR_RUN", RUN_SUBDIR)
paths = dm.paths_for_run(RUN_SUBDIR)
_, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
pos_idx = {v: i for i, v in enumerate(df["Vertex"])}
X = df[["Inf.Pos.1", "Inf.Pos.2", "Inf.Pos.3"]].to_numpy(dtype=np.float64)


**Sample positives and random negatives**

Draws random edges and random vertex pairs that are *not* edges. **Interpretation:** negatives are uniform over allowed pairs—not degree-matched controls—so differences are conservative and mainly qualitative.

In [ ]:
def dist_raw(i, j):
    d = X[i] - X[j]
    return float(np.sqrt(np.dot(d, d)))


def dist_sphere(i, j):
    xi = X[i] / (np.linalg.norm(X[i]) + 1e-12)
    xj = X[j] / (np.linalg.norm(X[j]) + 1e-12)
    cosang = float(np.clip(np.dot(xi, xj), -1.0, 1.0))
    return float(np.arccos(cosang))


rng = np.random.default_rng(2)
verts = list(G.nodes())
edges = list(G.edges())
n_pos = min(5000, len(edges))
n_neg = min(5000, len(edges))
pos_pairs = [edges[i] for i in rng.choice(len(edges), size=n_pos, replace=False)]

neg = []
edge_set = set(tuple(sorted(e)) for e in G.edges())
while len(neg) < n_neg:
    a, b = rng.choice(verts, size=2, replace=False)
    if a == b:
        continue
    key = tuple(sorted((a, b)))
    if key in edge_set:
        continue
    neg.append((a, b))


**Overlay distributions**

Normalized histograms of angular distance. **Interpretation:** a left shift of the blue (true edges) relative to orange suggests interacting pairs tend to align more closely on the sphere after per-vertex normalization—expected for a meaningful embedding, but magnitude can be subtle.

In [ ]:
d_pos = [dist_sphere(pos_idx[u], pos_idx[v]) for u, v in pos_pairs]
d_neg = [dist_sphere(pos_idx[u], pos_idx[v]) for u, v in neg]

plt.figure(figsize=(6, 4))
plt.hist(d_pos, bins=50, alpha=0.55, density=True, label="true edges")
plt.hist(d_neg, bins=50, alpha=0.55, density=True, label="random non-edges")
plt.xlabel("great-circle distance (normalized raw directions)")
plt.ylabel("density")
plt.title("Crude geometry sanity (not Mercator likelihood)")
plt.legend()
plt.tight_layout()
plt.show()


**Numeric summary**

Prints mean great-circle angle (in radians) for positives vs negatives. **Interpretation:** if true edges are systematically *lower* than random pairs, a purely geometric neighborhood signal exists—this still does **not** validate Mercator’s probabilistic model or replace proper link prediction.

In [ ]:
# Summary: mean distance (lower might be 'easier' for naive geometry)
print("mean sphere-angle true:", float(np.mean(d_pos)))
print("mean sphere-angle neg :", float(np.mean(d_neg)))
